# S0 dump: pointer-A errors on Omni eval (1x T4)

Scores released Soro-Kev-4B on build_v2 test/selection/eval_zeroshot. Writes error jsonl + bucket summary. One GPU.

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/Soro-Boro"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "upp-p0", "https://github.com/nisaral/Soro-Boro.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "origin", "upp-p0"], check=False)
os.chdir(REPO)
os.environ["PYTHONPATH"] = "veyra/python/src:veyra/training"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!pip -q install peft bitsandbytes huggingface_hub
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import time
from huggingface_hub import snapshot_download
from huggingface_hub.errors import HfHubHTTPError, LocalEntryNotFoundError
ckpt = None
for attempt in range(8):
    try:
        ckpt = snapshot_download("nisaralll/soro-kev-4b")
        break
    except (HfHubHTTPError, LocalEntryNotFoundError, OSError) as err:
        wait = 20 * (attempt + 1)
        print(f"hub download failed ({err}); sleep {wait}s", flush=True)
        time.sleep(wait)
if ckpt is None:
    raise SystemExit("could not download nisaralll/soro-kev-4b")
data = "veyra/omni_data/eval_splits"
if not os.path.isdir(data):
    data = "/kaggle/input/soro-omni-eval-splits"
print("ckpt", ckpt, "data", data, os.listdir(data), flush=True)
rc = subprocess.run(
    [sys.executable, "veyra/training/dump_a_errors.py", "--ckpt", ckpt, "--data-root", data, "--out", "/kaggle/working/s0_a_errors"],
    check=False,
)
print("dump rc", rc.returncode, flush=True)
if rc.returncode != 0:
    raise SystemExit(rc.returncode)

In [ ]:
import json, pathlib
p = pathlib.Path("/kaggle/working/s0_a_errors/summary.json")
print(p.read_text() if p.exists() else "no summary")